# CAFE ? Counterfactually Verified Explanation Demo

This notebook demonstrates the complete CAFE pipeline on one FaceForensics++ C23 video.

**Pipeline:** frozen detector ? candidate explanation generation ? counterfactual intervention ? control verification ? final explanation or abstention.

In [ ]:
import sys
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import cv2

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cafe.preprocessing import load_or_build_cache
from cafe.landmarks import load_or_build_landmark_cache
from cafe.detector.base_detector import BaseDetector
from cafe.interventions import apply_intervention
from cafe.candidates import generate_candidates
from cafe.utils.config import load_config

VIDEO_ID = 'Deepfakes_000_003'
VIDEO_PATH = ROOT / 'dataset/fake/Deepfakes/000_003.mp4'
CONFIG_PATH = ROOT / 'config.yaml'

print('Project root:', ROOT)
print('Video:', VIDEO_PATH)

## 1. Load configuration and preprocessed frames

In [ ]:
config = load_config(CONFIG_PATH)
config['paths'] = {k: str(ROOT / v) for k, v in config['paths'].items()}
cache = load_or_build_cache(VIDEO_ID, video_path=str(VIDEO_PATH))
landmarks = load_or_build_landmark_cache(VIDEO_ID)

faces = cache['faces']
print('Face crops:', faces.shape)
print('Sampled frames:', len(cache['index']))

## 2. Frozen detector prediction

In [ ]:
detector = BaseDetector()
per_frame_scores = detector.score_frames(faces)
original_score = float(np.mean(per_frame_scores))
prediction = 'DEEPFAKE' if original_score >= 0.5 else 'REAL'

print(f'Detector score: {original_score:.4f}')
print(f'Prediction: {prediction}')

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(np.arange(len(per_frame_scores)), per_frame_scores, marker='o', markersize=3)
plt.xlabel('Sampled frame position')
plt.ylabel('Detector score')
plt.title('Frozen detector per-frame scores')
plt.grid(alpha=0.25)
plt.show()

## 3. Generate candidate explanations

In [ ]:
candidates = generate_candidates(VIDEO_ID, config)

for i, c in enumerate(candidates):
    inner = c.get('candidate', c)
    print(
        f'Candidate {i + 1}: '
        f"cue={inner['cue']}, "
        f"original_interval={inner['interval']}, "
        f"sampled_interval={inner['sampled_interval']}"
    )

## 4. Counterfactual intervention

In [ ]:
# Demonstrate the first generated candidate.
candidate = candidates[0]
inner = candidate.get('candidate', candidate)
cue = inner['cue']
sampled_interval = tuple(inner['sampled_interval'])

modified_faces = apply_intervention(
    faces,
    landmarks,
    cue,
    sampled_interval,
    strength=None,
)

modified_scores = detector.score_frames(modified_faces)
modified_score = float(np.mean(modified_scores))
delta = original_score - modified_score

print(f'Cue: {cue}')
print(f'Sampled interval: {sampled_interval}')
print(f'Original score: {original_score:.6f}')
print(f'Counterfactual score: {modified_score:.6f}')
print(f'Effect (delta): {delta:.6f}')

In [ ]:
positions = np.linspace(sampled_interval[0], sampled_interval[1], 8).round().astype(int)
positions = np.clip(positions, 0, len(faces) - 1)

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
for j, p in enumerate(positions):
    axes[0, j].imshow(cv2.cvtColor(faces[p], cv2.COLOR_BGR2RGB))
    axes[0, j].set_title(f'{p}')
    axes[0, j].axis('off')
    axes[1, j].imshow(cv2.cvtColor(modified_faces[p], cv2.COLOR_BGR2RGB))
    axes[1, j].set_title(f'{p}')
    axes[1, j].axis('off')

axes[0, 0].set_ylabel('Original')
axes[1, 0].set_ylabel('Counterfactual')
plt.tight_layout()
plt.show()

## 5. Control verification

In [ ]:
# The saved batch result is used here so the notebook displays
# the exact control distribution and final CAFE decision.
result_path = ROOT / 'results/runs/Deepfakes_000_003__real.json'
with open(result_path, 'r', encoding='utf-8') as f:
    result = json.load(f)

print(f"Video: {result['video_id']}")
print(f"Condition: {result['condition']}")
print(f"Final detector score: {result['original_score']:.6f}")

for i, record in enumerate(result['candidates']):
    inner = record['candidate']
    print(
        f"Candidate {i + 1}: cue={inner['cue']}, "
        f"interval={inner['interval']}, "
        f"delta={record['candidate_effect']:.6f}, "
        f"tau={record['tau']:.6f}, "
        f"p={record['p_value']:.4f}, "
        f"supported={record['supported']}"
    )

In [ ]:
# Display the exact control distribution for the selected saved candidate.
selected = result['candidates'][2]
controls = np.asarray(selected['control_effects'], dtype=float)
candidate_delta = float(selected['candidate_effect'])
tau = float(selected['tau'])

plt.figure(figsize=(9, 4))
plt.hist(controls, bins=10, alpha=0.7)
plt.axvline(candidate_delta, linewidth=2, label='Candidate delta')
plt.axvline(tau, linestyle='--', linewidth=2, label='Tau')
plt.xlabel('Detector score effect (delta)')
plt.ylabel('Number of controls')
plt.title('Candidate effect versus control distribution')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 6. Final CAFE explanation

In [ ]:
selected_inner = selected['candidate']
if selected['supported']:
    print('VERIFIED EXPLANATION')
    print(f"Cue: {selected_inner['cue']}")
    print(f"Interval: frames {selected_inner['interval'][0]}?{selected_inner['interval'][1]}")
    print(f"Effect: {selected['candidate_effect']:.6f}")
    print(f"Control threshold: {selected['tau']:.6f}")
    print(f"p-value: {selected['p_value']:.6f}")
else:
    print('ABSTAIN')
    print('No candidate explanation exceeded its control-derived threshold.')